# Notebook 01: Data Collection

**Project:** Predicting Property-Level Housing Violation Risk in Boston
**Group:** Mengxing Wang, Jiacheng He, Xiao Luo, Renwei Li

## Purpose

This notebook documents and verifies the project data sources. It does not clean records, create analytical features, visualize patterns, or train models.

The project models **one Boston residential property per row** and predicts whether the property has been associated with a housing code violation. Sources are:

1. **Boston Building & Property Violations** - violation events used as the *label source*. Each violation is later snapped to the nearest residential property in notebook 02.
2. **Boston Property Assessment Data** - parcel-level structural attributes (year built, value, area, condition, etc.). These are the *modelling rows*.
3. **Boston Neighborhood Boundaries (GeoJSON)** - polygons used to assign each violation and each property to a Boston neighborhood.
4. **Boston 2025 Neighborhood Population Estimates** - context feature.
5. **Code Violations Data Dictionary** - lookup for violation codes.

Main outputs from this notebook:

- `data/external/neighborhood_population.csv`
- `data/processed/dataset_summary.json`

In [ ]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
from importlib.util import find_spec

current_dir = Path.cwd().resolve()
PROJECT_ROOT = current_dir if (current_dir / 'data').exists() else current_dir.parent
RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
EXTERNAL_DIR = PROJECT_ROOT / 'data' / 'external'

for d in [RAW_DIR, PROCESSED_DIR, EXTERNAL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Directory structure verified.')
print(f'Project root: {PROJECT_ROOT}')

## 1. Data Inventory

This table records which local files are expected, what role each file plays, and whether it should be committed to GitHub.

In [ ]:
data_inventory = pd.DataFrame([
    {
        'dataset': 'Building and Property Violations',
        'path': RAW_DIR / 'violations.csv',
        'role': 'Raw core violation records',
        'commit_policy': 'commit or provide clear download instructions',
    },
    {
        'dataset': '2025 Neighborhood Population Estimates',
        'path': EXTERNAL_DIR / 'boston_population_estimates_2025_neighborhood_level.csv',
        'role': 'Raw population source for neighborhood context',
        'commit_policy': 'commit',
    },
    {
        'dataset': 'Neighborhood Population Lookup',
        'path': EXTERNAL_DIR / 'neighborhood_population.csv',
        'role': 'Small generated lookup table used downstream',
        'commit_policy': 'commit',
    },
    {
        'dataset': 'Property Assessment',
        'path': EXTERNAL_DIR / 'boston_property_assessment_data.csv',
        'role': 'Large raw parcel-level housing stock source',
        'commit_policy': 'local only; ignored because file is large',
    },
    {
        'dataset': 'Neighborhood Structural Features',
        'path': EXTERNAL_DIR / 'neighborhood_structural_features.csv',
        'role': 'Small generated property feature table used downstream',
        'commit_policy': 'commit',
    },
    {
        'dataset': 'Code Violations Data Dictionary',
        'path': EXTERNAL_DIR / 'code-violations-data-dictionary.xlsx',
        'role': 'Official code, ordinance, and fine lookup',
        'commit_policy': 'commit',
    },
])

data_inventory['exists'] = data_inventory['path'].apply(lambda p: p.exists())
data_inventory['size_mb'] = data_inventory['path'].apply(
    lambda p: round(p.stat().st_size / 1024**2, 3) if p.exists() else np.nan
)
data_inventory['path'] = data_inventory['path'].astype(str)
data_inventory


## 2. Verify Raw Violations Dataset

The raw violations CSV is the core project dataset. This section verifies schema, date coverage, duplicate case numbers, status values, and city labels. Cleaning happens in Notebook 02.

In [ ]:
violations_path = RAW_DIR / 'violations.csv'
if not violations_path.exists():
    raise FileNotFoundError(f'Missing raw violations file: {violations_path}')

df = pd.read_csv(violations_path, low_memory=False)
df['status_dttm'] = pd.to_datetime(df['status_dttm'], errors='coerce')
valid_dates = df['status_dttm'].dropna()

required_violation_cols = [
    'case_no', 'status_dttm', 'status', 'code', 'description',
    'violation_city', 'violation_zip', 'latitude', 'longitude'
]
missing_required_cols = [c for c in required_violation_cols if c not in df.columns]
if missing_required_cols:
    raise KeyError(f'Missing required violation columns: {missing_required_cols}')

violation_summary = {
    'rows': len(df),
    'columns': df.shape[1],
    'date_min': str(valid_dates.min()),
    'date_max': str(valid_dates.max()),
    'missing_status_dttm': int(df['status_dttm'].isna().sum()),
    'duplicate_case_no_groups': int(df.groupby('case_no').size().gt(1).sum()),
    'unique_codes': int(df['code'].nunique()),
    'unique_descriptions': int(df['description'].nunique()),
}

print(json.dumps(violation_summary, indent=2))
print('\nStatus counts:')
print(df['status'].value_counts(dropna=False))
print('\nTop 10 raw violation_city values:')
print(df['violation_city'].value_counts(dropna=False).head(10))
df.head(3)

## 3. Generate Neighborhood Population Lookup

The raw population file contains many demographic columns. Downstream notebooks only need a small lookup table:

```text
neighborhood, population_2025
```

Using 2025 population as a fixed contextual feature is a limitation, but it is acceptable for this project scope because the model is case-level unresolved-risk classification, not demographic forecasting.

In [ ]:
population_path = EXTERNAL_DIR / 'boston_population_estimates_2025_neighborhood_level.csv'

if not population_path.exists():
    raise FileNotFoundError(
        f'Missing population file: {population_path}. '
        'Download it from the Boston 2025 neighborhood population estimates page '
        'and save it in data/external/.'
    )

pop_raw = pd.read_csv(population_path)
pop_raw.columns = pop_raw.columns.str.strip()

name_col = next((c for c in ['name', 'neighborhood', 'Neighborhood'] if c in pop_raw.columns), None)
population_candidates = [
    'population_b03002_001e',
    'population_b01001_001e',
]
pop_col = next((c for c in population_candidates if c in pop_raw.columns), None)

if name_col is None:
    raise KeyError(f'Could not find a neighborhood name column. Available columns: {pop_raw.columns.tolist()}')

if pop_col is None:
    possible_cols = [c for c in pop_raw.columns if c.lower().startswith('population')]
    raise KeyError(
        'Could not find the expected total population column. '
        f'Population-like columns include: {possible_cols[:15]}'
    )

pop_df = pop_raw[[name_col, pop_col]].copy()
pop_df = pop_df.rename(columns={name_col: 'neighborhood', pop_col: 'population_2025'})
pop_df['population_2025'] = pd.to_numeric(pop_df['population_2025'], errors='coerce')
pop_df = pop_df.dropna(subset=['neighborhood', 'population_2025'])
pop_df['neighborhood'] = pop_df['neighborhood'].astype(str).str.strip()
pop_df['population_2025'] = pop_df['population_2025'].round().astype(int)
pop_df = pop_df.sort_values('neighborhood').reset_index(drop=True)

population_lookup_path = EXTERNAL_DIR / 'neighborhood_population.csv'
pop_df.to_csv(population_lookup_path, index=False)
print(f'Loaded population from: {population_path.name}')
print(f'Using columns: {name_col!r}, {pop_col!r}')
print(f'Saved {len(pop_df)} neighborhood population rows to {population_lookup_path}')
pop_df.head()

## 4. Verify Property Assessment Source

The property assessment file is a large local source file. Notebook 01 only verifies that it exists and previews the schema. Cleaning and aggregation happen in Notebook 02.

In [ ]:
property_path = EXTERNAL_DIR / 'boston_property_assessment_data.csv'
structural_path = EXTERNAL_DIR / 'neighborhood_structural_features.csv'

if property_path.exists():
    prop_sample = pd.read_csv(property_path, nrows=5, low_memory=False)
    print(f'Property assessment raw file found: {property_path}')
    print(f'Size: {property_path.stat().st_size / 1024**2:.2f} MB')
    print(f'Columns: {len(prop_sample.columns)}')
    print(prop_sample.columns.tolist())
    display(prop_sample.head())
else:
    print('Raw property assessment CSV not found locally.')
    print('Expected path:', property_path)

if structural_path.exists():
    structural_sample = pd.read_csv(structural_path)
    print(f'Processed structural feature table found: {structural_path}')
    print(f'Rows: {len(structural_sample)}')
    display(structural_sample.head())
else:
    print('Processed structural feature table not found yet. Run Notebook 02 after adding the raw property file.')

## 5. Verify Code Violations Data Dictionary

The code dictionary is a small official lookup table for violation codes, descriptions, ordinances, and fines. Notebook 01 verifies the file. Notebook 02 can later clean it and merge it into `violations_clean.csv`.

In [ ]:
code_dictionary_path = EXTERNAL_DIR / 'code-violations-data-dictionary.xlsx'

if not code_dictionary_path.exists():
    print('Code dictionary file not found yet.')
    print('Expected path:', code_dictionary_path)
else:
    print(f'Code dictionary found: {code_dictionary_path}')
    print(f'Size: {code_dictionary_path.stat().st_size / 1024**2:.3f} MB')

    if find_spec('openpyxl') is None:
        print('openpyxl is not installed, so the Excel preview is skipped.')
        print('The file can still remain in data/external; add openpyxl or export to CSV before merging it in Notebook 02.')
    else:
        code_dict_preview = pd.read_excel(code_dictionary_path, header=1)
        code_dict_preview = code_dict_preview.dropna(how='all')
        print(f'Rows previewed: {len(code_dict_preview)}')
        print(f'Columns: {code_dict_preview.columns.tolist()}')
        display(code_dict_preview.head(10))

## 6. Save Data Source Summary

This summary records source availability and high-level raw violations metadata. It is useful for README reproducibility notes and later validation checks.

In [ ]:
summary = {
    'total_violation_records': int(len(df)),
    'violation_columns': df.columns.tolist(),
    'date_min': str(valid_dates.min()),
    'date_max': str(valid_dates.max()),
    'missing_status_dttm': int(df['status_dttm'].isna().sum()),
    'duplicate_case_no_groups': int(df.groupby('case_no').size().gt(1).sum()),
    'unique_codes': int(df['code'].nunique()),
    'raw_violations_path': str(violations_path),
    'population_source_available': bool(population_path.exists()),
    'population_lookup_path': str(population_lookup_path),
    'property_assessment_source_available': bool(property_path.exists()),
    'structural_feature_table_available': bool(structural_path.exists()),
    'code_dictionary_available': bool(code_dictionary_path.exists()),
}

summary_path = PROCESSED_DIR / 'dataset_summary.json'
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

print(f'Summary saved to {summary_path}')
summary
